# EDA - Deceptive Ads Dataset (`conpro-bad-ads-data`)

### In the context of: Multimodal Scam Detection System

**Dataset:** Eric Zeng, Tadayoshi Kohno, and Franziska Roesner, "Bad News: Clickbait and Deceptive Ads on News and Misinformation Websites" in Workshop on Technology and Consumer Protection (ConPro ’20), 2020.  
**Source:** https://github.com/eric-zeng/conpro-bad-ads-data

**Why this dataset fits the project.** Our system detects likely scams in *multimodal* social media content (text, images-with-embedded-text, links) and exlpains *why* if flagged them. This dataset covers on of our five target scam types - **deceptive ads** - and is itself multimodal: each ad has an image (the ad creative), a second image (its landing page), URLs (links), and categorical metadata. That makes it a good testbed for the same text + image + link fusion the whole project is build on.

This notebook performs standard EDA plus the required **Volume / Velocity / Variety / Veracity** assessment, checks whether the **data types align with the project scope** (can the output be transformed into our goal output?), and sets up the **source/target split, preprocessing, cleaning, and train/dev/test split** needed to train a supervised model.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward from `start` until a repo marker is found, so paths
    resolve the same whether the notebook is launched from the repo root
    from notebooks/image/."""
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").exists() or (cand / ".git").exists():
            return cand
    return p # fallback: current working directory


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "bad_ads" / "raw"
CSV_PATH = RAW_DIR / "dataset.csv"

print("Repo root:", REPO_ROOT)
print("Data dir:", RAW_DIR)

Repo root: C:\Users\Erik\BUAs\Pre-masters\SECSAI-group-6
Data dir: C:\Users\Erik\BUAs\Pre-masters\SECSAI-group-6\data\bad_ads\raw


## 1. Download & load the data

In [2]:
if not CSV_PATH.exists():
    import urllib.request
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CSV_URL = "https://github.com/eric-zeng/conpro-bad-ads-data/raw/refs/heads/master/dataset.csv"
    print("dataset.csv not found - downloading.")
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

Shape: (2421, 11)


,ad_id,landing_page_id,name,sample_type,page_type,parent_url,landing_url,ad_platform,ad_platform_type,ad_screenshot,page_screenshot
0,346865,100457.0,Charities and PSAs,adjusted_news,home,https://www.foxbusiness.com/,https://find.globalmissingkids.org/poster/NCME...,Google,Display,screenshots/e9d228d2-5374-4508-86ef-d86f4951f4...,external_screenshots/aea4ef82-edee-49c7-9370-0...
1,322828,93270.0,Charities and PSAs,adjusted_news,home,http://www.eatingwell.com/,https://www.aniclasssettlement.com/notice?gcli...,Google,Display,screenshots/f2e3f87b-369f-489f-9738-8cd2f9de6b...,external_screenshots/3e79b6c9-b7f6-4c68-b319-1...
2,409214,NaN,Charities and PSAs,adjusted_news,article,https://www.greenvilleonline.com/story/news/20...,NaN,Google,Display,screenshots/7facadb1-900a-4192-a92b-8a84349fa1...,external_screenshots/71a7f444-8534-45de-8e7d-9...
3,308364,88964.0,Charities and PSAs,adjusted_news,home,https://movieweb.com/,http://www.takataairbaginjurytrust.com/?utm_so...,Google,Display,screenshots/dc20d0e4-30e1-48e7-b940-43693168f9...,external_screenshots/f220ef3a-6153-4e79-a8d8-6...
4,346865,100457.0,Charities and PSAs,adjusted_news,home,https://www.foxbusiness.com/,https://find.globalmissingkids.org/poster/NCME...,Google,Display,screenshots/e9d228d2-5374-4508-86ef-d86f4951f4...,external_screenshots/aea4ef82-edee-49c7-9370-0...


## 2. Volume - *how much data is there?*

We look a the the number of records, the columns, in-memory size, and (importantly for a multimodal task) how many image files back the table.

In [3]:
print("Rows (ads):", len(df))
print("Columns:", df.shape[1])
print("Memory (MB): %.2f" % (df.memory_usage(deep=True).sum() / 1e6))
print("CSV size (KB): %.0f" % (CSV_PATH.stat().st_size / 1024))

Rows (ads): 2421
Columns: 11
Memory (MB): 1.95
CSV size (KB): 904


**Volume notes.** The labelled table is small - ~2.4k ads (the authors crawled ~55k, hand-labelled ~5.4k, and released the 2,421 that had usable content). Small volume is fine for our small-scale goal, but it limits how many parameters a model can learn from scratch, so we will lean on **pre-trained** vision/text encoders and fine-tune, rather than train large models end-to-end. The heavier part of the volume is the **image payload** (screenshots), hosted separately and downloaded once.

## 3. Velocity - *how fast does data arrive / change?*

In [4]:
# The crawl window is documented in the dataset README, not in the CSV.
print("Collection window: 15-19 January 2020 (one-time crawl).")
print("The CSV has not timestamp column, confirming this is a static snapshot:")
print([c for c in df.columns if "time" in c.lower() or "date" in c.lower()] or "  -> no time/date columns")

Collection window: 15-19 January 2020 (one-time crawl).
The CSV has not timestamp column, confirming this is a static snapshot:
  -> no time/date columns


**Velocity notes.** This is a **static, batch** dataset (velocity = 0): a single snapshot from Jan 2020, with no streaming or updates. Two implications for us: (1) our project objective of *adapting* to *new scam types* cannot be validated on this data alone - the ad tactics here are from 2020 and are subject to **concept drift**; (2) for deployment we should treat this as a *training/benchmark* corpus and plan a separate, refreshable data path for live content. At training time we simulate "velocity" only through how quickly the model must return a verdict (our latency KPI), not thourgh the data source.

## 4. Variety - *how many form does the data take?*

In [5]:
dtypes = df.dtypes.to_frame("dtype")
dtypes["n_unique"] = df.nunique()
dtypes["n_missing"] = df.isna().sum()
dtypes

,dtype,n_unique,n_missing
ad_id,int64,2162,0
landing_page_id,float64,1381,832
name,str,14,0
sample_type,str,3,0
page_type,str,2,0
parent_url,str,384,0
landing_url,str,1114,832
ad_platform,str,9,494
ad_platform_type,str,2,494
ad_screenshot,str,2162,0


In [6]:
modality = {
    "ad_id":            "identifier (int)",
    "landing_page_id":  "identifier (int, nullable)",
    "name":             "categorical  -> TARGET label",
    "sample_type":      "categorical  (news / adjusted_news / misinfo)",
    "page_type":        "categorical  (home / article)",
    "parent_url":       "link / text  (site the ad appeared on)",
    "landing_url":      "link / text  (where the ad leads)",
    "ad_platform":      "categorical  (Google, Taboola, ...)",
    "ad_platform_type": "categorical  (Display / Native)",
    "ad_screenshot":    "IMAGE path   (ad creative, text-in-image)",
    "page_screenshot":  "IMAGE path   (landing page, text-in-image)",
}
pd.Series(modality).to_frame("role / modality")

,role / modality
ad_id,identifier (int)
landing_page_id,"identifier (int, nullable)"
name,categorical -> TARGET label
sample_type,categorical (news / adjusted_news / misinfo)
page_type,categorical (home / article)
parent_url,link / text (site the ad appeared on)
landing_url,link / text (where the ad leads)
ad_platform,"categorical (Google, Taboola, ...)"
ad_platform_type,categorical (Display / Native)
ad_screenshot,"IMAGE path (ad creative, text-in-image)"


**Variety notes.** The dataset spans exactly the modalities our system fuses:

- **Images with embedded text** - the ad and landing-page screenshots. These are the core input and line up directly with our "image + OCR" pipeline.
- **Links** - `parent_url`, `landing_url`. Feed our URL/link-analysis component (domain, TLD, redirects, HTTPS).
- **Categorical metadata** - platform, native/display, page type. Useful features, though at inference on real social posts we may not have them. 

The one gap vs. our scope: there is no free-form **ad body text** field - that text lives *inside* the screenshots, so OCR is required to recover it. That is realistic and matches our "text embedded in images" requirement.

## 5. Veracity - *how trustworthy / clean is the data?*

In [7]:
print("Missing values by column:")
print(df.isna().sum()[lambda s: s > 0])

print("\nDuplicate ad_id values:", df["ad_id"].duplicated().sum(),
      "(same creative can appear on multiple pages)")

print("\nUnique parent sites:", df["parent_url"].nunique())
print("Sampling strate (sample_type):")
print(df["sample_type"].value_counts())

Missing values by column:
landing_page_id     832
landing_url         832
ad_platform         494
ad_platform_type    494
dtype: int64

Duplicate ad_id values: 259 (same creative can appear on multiple pages)

Unique parent sites: 384
Sampling strate (sample_type):
sample_type
news             914
adjusted_news    899
misinfo          608
Name: count, dtype: int64


**Veracity notes.**

- **Single annotator.** The README satates labelling was done by *one* coder, so there is **no inter-annotator agreement** to gauge label reliability.
- **Missing fields.** `landing_url` / `landing_page_id` are absent for ~34% of ads (the ad had no captured landing page), and `ad_platform` / `ad_platform_type` for ~20%.
- **Duplicates & leakage risk.** `ad_id` repeats (same ad on multiple pages) and ads cluster within 384 parent sites - we must avoid the **same ad/site landing in different splits** (see the split section).
- **Sampling bias.** Only top-ranked news and misinformation sites were crawled, so this is *not* a random sample of the web; class frequencies here need to match the wild.
- **Label naming.** The CSV uses short names (`Investing`, `Insurance`, `Product`) that differ from the paper's codebook wording - we normalise these below.

## Conclusion: dataset assessment

**What we assessed.** Sections 1–6 profiled the dataset along volume, velocity, variety, and veracity, and checked whether its output type aligns with our project goal (a scam verdict plus an explanation). The dataset is technically workable - the labels map cleanly to a binary target, and the images resolve correctly - but the decision below is driven by *project scope and fit*, not by any single data-quality failure.

**Context from our TA.** The advisor emphasised that this project is graded on
**well-thought, defensible decision-making**, not on building a novel or high-tech solution.

Two implications follow:

- A multimodal scam detector puts us in direct competition with large, well-resourced
  companies on a hard technical problem - the wrong place to demonstrate sound scoping.
- Adding explainability (XAI) as a system goal introduces a further objective on top of
  detection.

**Why we are not moving forward with this dataset.**

- **Text is embedded in images.** The ad copy is not a free-form text field; it lives inside the ad screenshots. Using this dataset in a text-focused, low-complexity scope would force an OCR/image pipeline.
- **Domain mismatch.** The data is display/native ads on news and misinformation *websites*, whereas our target is *social media* content. Usable as a proxy, but not a direct fit.
- **Volume limits granularity.** ~2.4k labelled ads support a binary target, but several scam-type classes have single-digit counts, ruling out reliable classification.
- **Labelling provenance.** A single expert annotator means no inter-annotator agreement to quantify label reliability, and the crawl is a one-time 2020 snapshot (concept-drift risk).

**When this dataset *would* be a good choice.** If the project scope were deliberately
multimodal - fusing ad image, embedded text (via OCR), and landing-page/URL signals - this dataset is a strong, well-documented fit and one of the few labelled resources for deceptive ads specifically. It would also suit a benchmark for image+text fusion. It is being set aside because of scope, not quality; a different scope would bring it back into consideration.